In [ ]:
# ── Notebook parameters ─────────────────────────────────────────
# Defaults for interactive use. extract_notebook_figures.py will
# override these via a cell injected right below this one.
#
# Add notebook-level globals here, e.g.:
# Analyzes an INR FPCA checkpoint. Produce one first with
#   python fpca.py +fpca_experiment=dwsnets_mnist_eulerian    (EXP_NAME='MNIST')
#   python fpca.py +fpca_experiment=implicit_zoo_cifar_eulerian  (EXP_NAME='CIFAR10')
# The run directory name is not predictable: `wandb-<run id>` with wandb enabled,
# otherwise a timestamp. List outputs/checkpoints/, pick your run, and use the
# highest step_*.pt (or best.pt) inside it.
CHECKPOINT_DIR  = "../outputs/checkpoints/XXX"
CHECKPOINT_FILE = "step_XXX.pt"
EXP_NAME = "MNIST"

# Visualization and reconstruction
CMAP = "Greys_r"
N_MEAN = 64
N_BATCH_SHOW = 12
N_VIS_ATOMS = 32
N_VAL = 4
N_RECON = 32
N_SHOW_LINEAR = 12
ATOM_TILE_SIZE = 0.55
RECON_ATOM_BUDGETS = [1, 4, 16, 64, 128, 256]
MOSAIC_SIDE = 7          # mosaic of the first MOSAIC_SIDE² prefix atoms

# Spectral diagnostics
N_SPEC = 200
SPECTRAL_ATOMS = 256
N_VIS_SPEC = 32
N_VIOLIN = 10


## 1  FPCA Checkpoint — Setup

Load a checkpoint produced by `fpca.py` with either the
`dwsnets_mnist_eulerian` or `implicit_zoo_cifar_eulerian` experiment config.
All architecture and data-path details are read from the saved `config.yaml`.

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "..")

In [ ]:

# ──────────────────────────────────────────────────────────────────────────────

import yaml
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from omegaconf import OmegaConf
from hydra.utils import instantiate

from infidictionary.datasets.inr import _eval_siren_batch
from infidictionary.neural_isometries import NeuralIsometry
from infidictionary.networks import NeuralField
from infidictionary.dictionaries import InfiDictionary
from infidictionary.domain_samplers import SquareSampler
from infidictionary.utils import (
    norm2, nested_projection_energies, prefix_weighted_energy, project_onto_span,
)
from notebook_helpers import (
    plot_atom_mosaic, plot_atoms_comparison, show_field,
    psnr_matrix, plot_psnr_line_chart, plot_spectral_compactness,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

with open(f"{CHECKPOINT_DIR}/config.yaml") as f:
    conf = OmegaConf.create(yaml.safe_load(f))

# Config paths are relative to the repository root, not the notebook kernel cwd.
_repo_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "infidictionary").is_dir()),
    None,
)
if _repo_root is None:
    raise RuntimeError("Could not locate the repository root from the notebook working directory")
if OmegaConf.select(conf, "ood_function_generator") is None:
    raise ValueError(
        "This checkpoint predates the explicit `ood_function_generator` config. "
        "Retrain it with the current INR FPCA experiment configuration."
    )

def resolve_dataset_paths(dataset_cfg):
    for path_key in ("data_dir", "extracted_dir"):
        value = dataset_cfg.get(path_key)
        if value is not None:
            path = Path(str(value))
            dataset_cfg[path_key] = str(path if path.is_absolute() else _repo_root / path)

resolve_dataset_paths(conf.function_generator)
resolve_dataset_paths(conf.ood_function_generator)

# ── Optionally cap the number of loaded INRs to speed things up ───────────────
# OmegaConf.update(conf, "function_generator.n_inrs", 500, merge=True)

neural_isometry: NeuralIsometry    = instantiate(conf.neural_isometry).to(device)
mean_function:   NeuralField       = instantiate(conf.mean_function).to(device)
initial_dictionary: InfiDictionary = instantiate(conf.initial_dictionary).to(device)
function_generator                 = instantiate(conf.function_generator)

ckpt = torch.load(
    f"{CHECKPOINT_DIR}/{CHECKPOINT_FILE}", map_location=device, weights_only=False
)
neural_isometry.load_state_dict(ckpt["models"]["neural_isometry"])
mean_function.load_state_dict(ckpt["models"]["mean_function"])
if "initial_dictionary" in ckpt["models"]:
    initial_dictionary.load_state_dict(ckpt["models"]["initial_dictionary"])

neural_isometry.eval()
mean_function.eval()

CHANNELS_DIM = int(conf.initial_dictionary.get("num_channels", 1))
dictionary_name = type(initial_dictionary).__name__.removesuffix("Dictionary")
raw_basis_label = f"Raw {dictionary_name}"
transported_basis_label = "Learned Dictionary"

print(f"Loaded  : {CHECKPOINT_DIR}/{CHECKPOINT_FILE}  epoch={ckpt['epoch']}")
print(f"Metric  : {ckpt.get('metric', float('nan')):.4f}  |  best: {ckpt.get('best_metric', float('nan')):.4f}")
print(f"Dataset : {type(function_generator).__name__}  n_inrs={function_generator.n_inrs}  C={CHANNELS_DIM}")

# ── Ordered prefix the isometry was trained on ────────────────────────────────
# The dictionary is only probed through its ordered prefix e_1..e_K (descending
# PMF). Training pushes the top-`num_atoms` prefix through the isometry, and
# learned atom k is row k of the pushed prefix. The rotations are built from its
# first R rows; R is the top-level `rank` key of the run config and is passed to
# every pushforward (it is not a constructor argument), which needs K >= R. By
# prefix faithfulness the first K pushed rows do not depend on how many further
# atoms are appended, so a longer prefix extends the learned basis consistently.
NUM_ATOMS   = int(conf.num_atoms)
RANK        = int(conf["rank"])
PREFIX_PMFS = initial_dictionary.get_prefix_pmfs(NUM_ATOMS).to(device)
print(f"Prefix  : K={NUM_ATOMS} atoms, rank R={RANK}, layers={getattr(neural_isometry, 'num_layers', '-')}")

from notebook_helpers import plot_recon_grid

def raw_prefix(coords: torch.Tensor, num_atoms: int) -> torch.Tensor:
    """The first ``num_atoms`` dictionary atoms e_1..e_K at ``coords``: (K, N, C)."""
    return initial_dictionary.get_prefix(coords, num_atoms)

def learned_prefix(coords: torch.Tensor, num_atoms: int) -> torch.Tensor:
    """The first ``num_atoms`` learned atoms, rows of T_R(e_1..e_K) at ``coords``: (K, N, C).

    Pushes a prefix of length max(K, R), since the pushforward needs K >= R,
    and keeps the first K rows.
    """
    lad = torch.zeros(coords.shape[0], device=coords.device)
    prefix = initial_dictionary.get_prefix(coords, max(num_atoms, RANK))
    _, _, pushed = neural_isometry.pushforward(coords, lad, prefix, rank=RANK)
    return pushed[:num_atoms]

def reconstruct_in_basis(
    functions: torch.Tensor, mean: torch.Tensor, basis: torch.Tensor,
) -> torch.Tensor:
    """Orthogonal projection of centred ``functions`` (B, N, C) onto span(``basis``), plus the mean.

    ``basis`` (k, N, C) need not be orthonormal: ``project_onto_span`` solves
    with its Gram matrix. For an orthonormal prefix (Fourier happens to be one)
    this is Σ_j ⟨f - μ, e_j⟩ e_j. With ``basis = learned_prefix(coords, k)`` it
    projects onto span(T e_1, …, T e_k).
    """
    projected, _ = project_onto_span(functions - mean.unsqueeze(0), basis)
    return projected + mean.unsqueeze(0)

def print_threshold_table(cum_raw_mean, cum_learned_mean, fracs=(0.50, 0.75, 0.90)):
    """Atoms needed to reach each fraction of ‖f - μ‖², from mean cumulative-fraction curves."""
    def need(cum, frac):
        n = int((cum < frac).sum()) + 1
        return n if n <= len(cum) else None   # None: not reached within the prefix
    print(f"\n{'Threshold':>10}  {dictionary_name:>10}  {'Learned':>10}  {'gain':>8}")
    for frac in fracs:
        nf, nl = need(cum_raw_mean, frac), need(cum_learned_mean, frac)
        cells = [f"{n:>9d}" if n else f"{'>' + str(len(cum_raw_mean)):>9}" for n in (nf, nl)]
        gain = f"{(nf - nl) / nf * 100:>+7.1f}%" if nf and nl else f"{'n/a':>8}"
        print(f"{frac*100:>9.0f}%  {cells[0]}  {cells[1]}  {gain}")

def prefix_mosaic_indices(side: int) -> torch.Tensor:
    """Layout indices for ``plot_atom_mosaic`` over the first ``side²`` prefix atoms.

    Rows hold consecutive prefix positions, starting from the bottom row; the
    1-based prefix position of a tile is its row label plus its column label.
    """
    pos = torch.arange(side * side)
    return torch.stack([(pos // side) * side, pos % side + 1, torch.zeros_like(pos)], dim=1)

def eval_inrs_at_coords(fg, coords_01, num_elements, seed, dev):
    """Randomly sample num_elements SIRENs and evaluate at coords in [0,1]² → (num_elements, N, C)."""
    n_total = fg.weights[0].shape[0]
    rng = np.random.default_rng(seed)
    idx = rng.choice(n_total, size=min(num_elements, n_total), replace=False)
    sc = torch.stack([
        2.0 * coords_01[:, 0] - 1.0,
        1.0 - 2.0 * coords_01[:, 1],
    ], dim=-1).to(dev)
    w  = [W[idx].to(dev) for W in fg.weights]
    b  = [bb[idx].to(dev) for bb in fg.biases]
    from infidictionary.datasets.inr import _eval_siren_batch
    with torch.no_grad():
        return _eval_siren_batch(sc, w, b, fg.omega_0, fg.omega)

## 2  Sample Batch Gallery

A quick look at what the loaded `function_generator` actually contains.
MNIST uses hexbin (C=1, grayscale); CIFAR-10 uses RGB scatter (C=3).

In [ ]:
function_generator.reset_buffer()
c_b, v_b = function_generator.get_batch(N_BATCH_SHOW)   # coords in [0,1]²
xy_b = c_b.detach().cpu().numpy()                  # display in [0,1]²

fig, axes = plt.subplots(2, N_BATCH_SHOW // 2, figsize=(18, 6))

if CHANNELS_DIM == 1:
    for i, ax in enumerate(axes.flat):
        vals_i = v_b[i, :, 0].detach().cpu().numpy()
        ax.hexbin(xy_b[:, 0], xy_b[:, 1], C=vals_i,
                  gridsize=14, cmap='gray', reduce_C_function=np.mean)
        ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
        ax.set_aspect('equal'); ax.axis('off')
    plt.suptitle(f'Sample batch — {type(function_generator).__name__} (hexbin)', y=1.01)
else:
    for i, ax in enumerate(axes.flat):
        rgb = v_b[i].detach().cpu().numpy()   # (N, 3)
        rgb_norm = (rgb - rgb.min(0)) / (rgb.max(0) - rgb.min(0) + 1e-8)
        ax.scatter(xy_b[:, 0], xy_b[:, 1], c=rgb_norm, s=50, alpha=0.7)
        ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
        ax.set_aspect('equal'); ax.axis('off')
    plt.suptitle(f'Sample batch — {type(function_generator).__name__} (RGB scatter)', y=1.01)

plt.tight_layout()
plt.show()

### Learned Mean Function $\mu(x)$

Co-trained with the isometry; Minimizes MSE to the dataset mean.
The isometry then diagonalises the covariance of the *zero-centred* data.

In [ ]:
coords_mean = SquareSampler(stratified=True, add_noise=False).sample(N_MEAN).to(device)
with torch.no_grad():
    mean_vals = mean_function(coords_mean)   # (N, C)

fig, ax = plt.subplots(figsize=(4.2, 4.2))
show_field(ax, mean_vals, N_MEAN, title="Learned mean function", cmap=CMAP)
plt.tight_layout(); plt.show()

## 3  Learned Basis — Initial Atoms vs. Pushed-Forward Atoms

Take the ordered prefix $e_1, \dots, e_K$ of the configured initial dictionary
(atoms by descending PMF) and evaluate its pushforward on a regular grid. The
isometry is a stack of `num_layers` Cayley rotations built from the first
$R=$ `rank` atoms of the prefix; learned atom $k$ is row $k$ of the pushed prefix.
Grayscale for MNIST ($C=1$); RGB tiles for CIFAR ($C=3$).

The mosaic lays out the first `MOSAIC_SIDE`² prefix atoms row by row from the
bottom; a tile's prefix position is its row label plus its column label. If the
isometry has converged, the pushed-forward atoms should look like data-adapted
basis functions (e.g. edges, blobs, colour patches) rather than the initial atoms.

In [ ]:
coords_atoms = SquareSampler(stratified=True, add_noise=False).sample(N_VIS_ATOMS).to(device)

# Shortest prefix that covers both figures.
n_mosaic    = MOSAIC_SIDE * MOSAIC_SIDE
n_basis_vis = max(N_SHOW_LINEAR, n_mosaic)
print(f"Showing the first {n_basis_vis} prefix atoms of {dictionary_name} (C={CHANNELS_DIM})")

with torch.no_grad():
    atoms_src = raw_prefix(coords_atoms, n_basis_vis)       # (A, N, C)
    atoms_tgt = learned_prefix(coords_atoms, n_basis_vis)   # (A, N, C)

# ── 2-row linear comparison (prefix order, ν descending) ─────────────────────
plot_atoms_comparison(
    atoms_src.cpu(), atoms_tgt.cpu(),
    n_vis=N_VIS_ATOMS, n_show=N_SHOW_LINEAR, tile_size=ATOM_TILE_SIZE, cmap=CMAP,
    row_labels=(raw_basis_label, transported_basis_label),
    title=f"First {N_SHOW_LINEAR} {dictionary_name} prefix atoms (ν descending)",
)

# ── Mosaic of the first MOSAIC_SIDE² prefix atoms ────────────────────────────
mosaic_idx = prefix_mosaic_indices(MOSAIC_SIDE)
plot_atom_mosaic(atoms_src[:n_mosaic].cpu(), mosaic_idx,
                 title=f"Raw {dictionary_name} atoms (first {n_mosaic} prefix atoms)",
                 n_vis=N_VIS_ATOMS, tile_size=ATOM_TILE_SIZE, cmap=CMAP)
plot_atom_mosaic(atoms_tgt[:n_mosaic].cpu(), mosaic_idx,
                 title=f"Learned atoms from {dictionary_name} (first {n_mosaic} prefix atoms)",
                 n_vis=N_VIS_ATOMS, tile_size=ATOM_TILE_SIZE, cmap=CMAP)


## 4  Reconstruction — In-Distribution

Reconstruction in the configured initial dictionary vs. its learned transported basis, at several atom budgets $k$. Each reconstruction is the orthogonal projection of $f-\mu$ onto the span of the first $k$ atoms of the (raw or pushed) ordered prefix, computed through their Gram matrix so the atoms need not be orthonormal. PSNR (dB) is shown above each tile.

In [ ]:

coords_recon = SquareSampler(stratified=True, add_noise=False).sample(N_RECON).to(device)

val_in = eval_inrs_at_coords(function_generator, coords_recon, N_VAL, 42, device)
print(f"In-distribution signals: {tuple(val_in.shape)}")

n_atoms_list = list(dict.fromkeys(RECON_ATOM_BUDGETS))
print(f"Reconstruction atom budgets: {n_atoms_list}")

# One prefix per basis, long enough for the largest budget; budget k uses its first k rows.
with torch.no_grad():
    mean_recon          = mean_function(coords_recon)
    basis_raw_recon     = raw_prefix(coords_recon, max(n_atoms_list))
    basis_learned_recon = learned_prefix(coords_recon, max(n_atoms_list))

    initial_recons_in = [
        reconstruct_in_basis(val_in, mean_recon, basis_raw_recon[:budget]).cpu()
        for budget in n_atoms_list
    ]
    learned_recons_in = [
        reconstruct_in_basis(val_in, mean_recon, basis_learned_recon[:budget]).cpu()
        for budget in n_atoms_list
    ]

metrics_in = plot_recon_grid(
    methods=[("Learned", learned_recons_in), (raw_basis_label, initial_recons_in)],
    val_imgs_cpu=val_in.cpu(),
    n_atoms_list=n_atoms_list,
    n=N_RECON,
    cmap=CMAP,
)

## 5  Reconstruction — Out-of-Distribution

The checkpoint configuration explicitly supplies `ood_function_generator` and `ood_label`. The notebook instantiates that configuration directly; `ood_fg` is reused by the spectral compactness section below.

In [ ]:
ood_fg = instantiate(conf.ood_function_generator)
ood_label = str(conf.ood_label)
print(f"OOD: {ood_label}  n_inrs={ood_fg.n_inrs}")

In [ ]:
if ood_fg is not None:
    val_ood = eval_inrs_at_coords(ood_fg, coords_recon, N_VAL, 0, device)
    print(f"OOD signals: {tuple(val_ood.shape)}")

    with torch.no_grad():
        initial_recons_ood = [
            reconstruct_in_basis(val_ood, mean_recon, basis_raw_recon[:budget]).cpu()
            for budget in n_atoms_list
        ]
        learned_recons_ood = [
            reconstruct_in_basis(val_ood, mean_recon, basis_learned_recon[:budget]).cpu()
            for budget in n_atoms_list
        ]

    metrics_ood = plot_recon_grid(
        methods=[("Learned", learned_recons_ood), (raw_basis_label, initial_recons_ood)],
        val_imgs_cpu=val_ood.cpu(),
        n_atoms_list=n_atoms_list,
        n=N_RECON,
        cmap=CMAP,
    )

## PSNR vs. Number of Atoms

Bar chart summarising reconstruction quality for every signal and truncation level, with one panel per signal. In-distribution (top) and OOD (bottom, if available).

In [ ]:
psnr_in_F = psnr_matrix(initial_recons_in, val_in.cpu())  # (N_VAL, K)
psnr_in_L = psnr_matrix(learned_recons_in, val_in.cpu())

in_methods = [("Learned", psnr_in_L), (raw_basis_label, psnr_in_F)]

if ood_fg is not None:
    psnr_ood_F = psnr_matrix(initial_recons_ood, val_ood.cpu())
    psnr_ood_L = psnr_matrix(learned_recons_ood, val_ood.cpu())
    ood_methods = [("Learned", psnr_ood_L), (raw_basis_label, psnr_ood_F)]
else:
    ood_methods = None

plot_psnr_line_chart(
    in_methods=in_methods,
    n_atoms_list=n_atoms_list,
    ood_methods=ood_methods,
)


## 6  Spectral Compactness — In-Distribution

For each of `N_SPEC` training-split SIRENs, measure how much of the energy
$\|f-\mu\|^2$ lies in the span of the first $k$ atoms of the ordered prefix
($\nu(k)$, the prior PMF, descending), for every $k \le$ `SPECTRAL_ATOMS`:

$$E_k^F = \|P_{\langle e_1,\dots,e_k\rangle}(f - \mu)\|^2, \qquad
  E_k^L = \|P_{\langle T e_1,\dots,T e_k\rangle}(f - \mu)\|^2,$$

where $T e_k$ is row $k$ of the pushed prefix on the same grid. All $k$ come
from one Gram–Schmidt pass in prefix order, so the prefix need not be
orthonormal. The curves show $E_k/\|f-\mu\|^2$; the per-atom panel shows the
increment $E_k - E_{k-1}$, the energy atom $k$ adds beyond atoms $1..k-1$
(for an orthonormal prefix, such as Fourier, this is $\langle f-\mu, e_k\rangle^2$).
We also report the training objective, the PMF-weighted energy
$\sum_{k\le K}\nu(k)\,\langle f-\mu, T e_k\rangle^2$ over the
$K=$ `num_atoms` prefix. If FPCA has converged, the learned basis should pack
more energy into the first few atoms.

In [ ]:
coords_spec = SquareSampler(stratified=True, add_noise=False).sample(N_VIS_SPEC).to(device)
lad_spec    = torch.zeros(coords_spec.shape[0], device=device)

A_spec  = SPECTRAL_ATOMS
spec_nu = initial_dictionary.get_prefix_pmfs(A_spec).cpu()
print(f"{A_spec} prefix atoms  ν ∈ [{spec_nu[-1]:.2e}, {spec_nu[0]:.2e}]")

# Evaluate training SIRENs on the fixed grid
imgs_in = eval_inrs_at_coords(function_generator, coords_spec, N_SPEC, 0, device)
print(f"In-distribution signals: {tuple(imgs_in.shape)}")

In [ ]:
with torch.no_grad():
    mu_spec = mean_function(coords_spec)                                     # (N, C)
    ctr_in  = imgs_in - mu_spec.unsqueeze(0)                                 # (n_in, N, C)

    # One prefix per basis, covering both the spectrum and the training objective.
    n_spec_prefix   = max(A_spec, NUM_ATOMS)
    prefix_raw      = raw_prefix(coords_spec, n_spec_prefix)                 # e_1..e_n
    prefix_learned  = learned_prefix(coords_spec, n_spec_prefix)             # T e_1..T e_n

    # E_k = ‖P_k (f - μ)‖² for every prefix length k <= A, and ‖f - μ‖².
    E_F_in   = nested_projection_energies(ctr_in, prefix_raw[:A_spec], lad_spec).cpu()      # (n_in, A)
    E_L_in   = nested_projection_energies(ctr_in, prefix_learned[:A_spec], lad_spec).cpu()  # (n_in, A)
    tot_in   = norm2(ctr_in, lad_spec).cpu()                                                # (n_in,)

    # Training objective: PMF-weighted energy over the K-atom prefix.
    E_prefix_raw_in = prefix_weighted_energy(
        ctr_in, prefix_raw[:NUM_ATOMS], PREFIX_PMFS, lad_spec).mean()
    E_prefix_learned_in = prefix_weighted_energy(
        ctr_in, prefix_learned[:NUM_ATOMS], PREFIX_PMFS, lad_spec).mean()

# Energy atom k adds beyond atoms 1..k-1 (⟨f - μ, e_k⟩² for an orthonormal prefix).
dE_F_in = torch.diff(E_F_in, dim=-1, prepend=torch.zeros_like(E_F_in[:, :1]))   # (n_in, A)
dE_L_in = torch.diff(E_L_in, dim=-1, prepend=torch.zeros_like(E_L_in[:, :1]))
print(f"Total energy ‖f-μ‖² (in-dist): {tot_in.mean():.4f}")
print(f"Energy in the first {A_spec} atoms (in-dist) — {dictionary_name}: {E_F_in[:, -1].mean():.4f}  "
      f"Learned: {E_L_in[:, -1].mean():.4f}")
print(f"PMF-weighted energy (training objective, K={NUM_ATOMS}, in-dist) — {dictionary_name}: "
      f"{E_prefix_raw_in:.4e}  Learned: {E_prefix_learned_in:.4e}")

In [ ]:
# Per-image cumulative fraction of total energy ‖P_k f‖² / ‖f‖²: (n_in, A)
cum_F_in_per_img = E_F_in / (tot_in[:, None] + 1e-15)
cum_L_in_per_img = E_L_in / (tot_in[:, None] + 1e-15)

cum_F_in_mean = cum_F_in_per_img.mean(dim=0).numpy()
cum_F_in_std  = cum_F_in_per_img.std(dim=0).numpy()
cum_L_in_mean = cum_L_in_per_img.mean(dim=0).numpy()
cum_L_in_std  = cum_L_in_per_img.std(dim=0).numpy()

x_in = np.arange(1, A_spec + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

plot_spectral_compactness(
    [
        dict(label=dictionary_name, xs=x_in, mean=cum_F_in_mean, std=cum_F_in_std, color='C0'),
        dict(label="Learned", xs=x_in, mean=cum_L_in_mean, std=cum_L_in_std, color='C3'),
    ],
    xlabel="atom rank  (ν descending)",
    xscale="log",
    axes=(axes[0], axes[1]),
)

# ── Per-atom mean ± std  (top N_VIOLIN atoms) ─────────────────────────────
ax  = axes[2]
nv  = min(N_VIOLIN, A_spec)
pos = np.arange(1, nv + 1)

mean_F_in = dE_F_in[:, :nv].mean(dim=0).numpy()
std_F_in  = dE_F_in[:, :nv].std(dim=0).numpy()
mean_L_in = dE_L_in[:, :nv].mean(dim=0).numpy()
std_L_in  = dE_L_in[:, :nv].std(dim=0).numpy()

ax.plot(pos, mean_F_in, 'o-', lw=1.5, ms=4, color='C0', label=dictionary_name)
ax.fill_between(pos, np.maximum(mean_F_in - std_F_in, 0), mean_F_in + std_F_in,
                alpha=0.25, color='C0')
ax.plot(pos, mean_L_in, 'o-', lw=1.5, ms=4, color='C3', label="Learned")
ax.fill_between(pos, np.maximum(mean_L_in - std_L_in, 0), mean_L_in + std_L_in,
                alpha=0.25, color='C3')
ax.set_xticks(pos)
ax.set_xticklabels([str(k) for k in pos], fontsize=7)
ax.set_xlabel("atom rank  (ν descending, with tiebreak)")
ax.set_ylabel("$E_k - E_{k-1}$  (mean ± 1 std)")
ax.set_title(f"Per-image energy  (top {nv} atoms)")
ax.legend(); ax.grid(True, alpha=0.3, axis="y")

plt.suptitle(
    f"Spectral compactness — In-distribution  "
    f"({imgs_in.shape[0]} training SIRENs, {A_spec} prefix atoms)",
    y=1.02,
)
plt.tight_layout()
plt.show()

# ── Summary table ──────────────────────────────────────────────────────────────
print_threshold_table(cum_F_in_mean, cum_L_in_mean)

## 7  Spectral Compactness — Out-of-Distribution

Load held-out SIRENs and repeat the spectral analysis:

| Training dataset | OOD source |
|---|---|
| `ImplicitZooMNISTDataset` | `mnist_png_testing*` (MNIST test split) |
| `ImplicitZooCIFARDataset` | `split='eval'` (10 000 held-out CIFAR-10 SIRENs) |

A well-trained FPCA basis should generalise: the learned representation should
remain compact on held-out data from the same distribution.

In [ ]:
if ood_fg is not None:
    imgs_ood = eval_inrs_at_coords(ood_fg, coords_spec, N_SPEC, 0, device)
    print(f"OOD signals for spectral: {tuple(imgs_ood.shape)}")

    with torch.no_grad():
        ctr_ood = imgs_ood - mu_spec.unsqueeze(0)                                # (n_ood, N, C)

        E_F_ood = nested_projection_energies(ctr_ood, prefix_raw[:A_spec], lad_spec).cpu()      # (n_ood, A)
        E_L_ood = nested_projection_energies(ctr_ood, prefix_learned[:A_spec], lad_spec).cpu()  # (n_ood, A)
        tot_ood = norm2(ctr_ood, lad_spec).cpu()                                                # (n_ood,)
        E_prefix_raw_ood = prefix_weighted_energy(
            ctr_ood, prefix_raw[:NUM_ATOMS], PREFIX_PMFS, lad_spec).mean()
        E_prefix_learned_ood = prefix_weighted_energy(
            ctr_ood, prefix_learned[:NUM_ATOMS], PREFIX_PMFS, lad_spec).mean()

    dE_F_ood = torch.diff(E_F_ood, dim=-1, prepend=torch.zeros_like(E_F_ood[:, :1]))   # (n_ood, A)
    dE_L_ood = torch.diff(E_L_ood, dim=-1, prepend=torch.zeros_like(E_L_ood[:, :1]))
    print(f"Total energy ‖f-μ‖² (OOD): {tot_ood.mean():.4f}")
    print(f"Energy in the first {A_spec} atoms (OOD) — {dictionary_name}: {E_F_ood[:, -1].mean():.4f}  "
          f"Learned: {E_L_ood[:, -1].mean():.4f}")
    print(f"PMF-weighted energy (training objective, K={NUM_ATOMS}, OOD) — {dictionary_name}: "
          f"{E_prefix_raw_ood:.4e}  Learned: {E_prefix_learned_ood:.4e}")

    # Per-image cumulative fraction of total energy ‖P_k f‖² / ‖f‖²: (n_ood, A)
    cum_F_ood_per_img = E_F_ood / (tot_ood[:, None] + 1e-15)
    cum_L_ood_per_img = E_L_ood / (tot_ood[:, None] + 1e-15)

    cum_F_ood_mean = cum_F_ood_per_img.mean(dim=0).numpy()
    cum_F_ood_std  = cum_F_ood_per_img.std(dim=0).numpy()
    cum_L_ood_mean = cum_L_ood_per_img.mean(dim=0).numpy()
    cum_L_ood_std  = cum_L_ood_per_img.std(dim=0).numpy()

    x_ood = np.arange(1, A_spec + 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

    plot_spectral_compactness(
        [
            dict(label=dictionary_name, xs=x_ood, mean=cum_F_ood_mean, std=cum_F_ood_std, color='C0'),
            dict(label="Learned", xs=x_ood, mean=cum_L_ood_mean, std=cum_L_ood_std, color='C3'),
        ],
        xlabel="atom rank  (ν descending)",
        xscale="log",
        axes=(axes[0], axes[1]),
    )

    # ── Per-atom mean ± std  (top N_VIOLIN atoms) ─────────────────────────
    ax  = axes[2]
    nv  = min(N_VIOLIN, A_spec)
    pos = np.arange(1, nv + 1)

    mean_F_ood = dE_F_ood[:, :nv].mean(dim=0).numpy()
    std_F_ood  = dE_F_ood[:, :nv].std(dim=0).numpy()
    mean_L_ood = dE_L_ood[:, :nv].mean(dim=0).numpy()
    std_L_ood  = dE_L_ood[:, :nv].std(dim=0).numpy()

    ax.plot(pos, mean_F_ood, 'o-', lw=1.5, ms=4, color='C0', label=dictionary_name)
    ax.fill_between(pos, np.maximum(mean_F_ood - std_F_ood, 0), mean_F_ood + std_F_ood,
                    alpha=0.25, color='C0')
    ax.plot(pos, mean_L_ood, 'o-', lw=1.5, ms=4, color='C3', label="Learned")
    ax.fill_between(pos, np.maximum(mean_L_ood - std_L_ood, 0), mean_L_ood + std_L_ood,
                    alpha=0.25, color='C3')
    ax.set_xticks(pos)
    ax.set_xticklabels([str(k) for k in pos], fontsize=7)
    ax.set_xlabel("atom rank  (ν descending, with tiebreak)")
    ax.set_ylabel("$E_k - E_{k-1}$  (mean ± 1 std)")
    ax.set_title(f"Per-image energy  (top {nv} atoms)")
    ax.legend(); ax.grid(True, alpha=0.3, axis="y")

    plt.suptitle(
        f"Spectral compactness — OOD  "
        f"({ood_label}, {imgs_ood.shape[0]} SIRENs, {A_spec} prefix atoms)",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

    # ── Summary table ──────────────────────────────────────────────────────────
    print_threshold_table(cum_F_ood_mean, cum_L_ood_mean)